## Computational Methods in Materials Science
@ Anuj Goyal (Aug-Dec 2026)

## Matrix–Vector Algebra:

**Outline**

* Build a **geometric picture** of $Ax = b$
* See why **orthogonal** and **orthonormal** columns make life easy
* See how a general matrix **stretches and rotates** a vector
* A matrix decomposed via **SVD**: rotate → stretch → rotate
* A matrix decomposed via **QR**: rotate → (stretch + shear)

---

### References

**Notes are inspired by the discussion in the reference**

* [L. N.Trefethen \& David Bau III, *Numerical Linear Algebra (Chapters 1 and 2)*](https://www.stat.uchicago.edu/~lekheng/courses/309/books/Trefethen-Bau.pdf)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, ToggleButtons

%matplotlib inline
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

# colour-blind-safe colours
BLUE, ORANGE, GREEN, RED, PURPLE, GRAY = "#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#8c8c8c"

In [ ]:
# ---------------- small drawing helpers used by all figures ----------------
def arrow(ax, vec, tail=(0, 0), color="k", label=None, lw=2.5, ls="-", alpha=1.0, offset=(0.15, 0.15)):
    "Draw the vector `vec` starting at `tail`; optional label near the tip."
    tail, vec = np.asarray(tail, float), np.asarray(vec, float)
    tip = tail + vec
    ax.annotate("", xy=tip, xytext=tail,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, ls=ls, alpha=alpha,
                                shrinkA=0, shrinkB=0, mutation_scale=16))
    if label:
        ax.text(tip[0] + offset[0], tip[1] + offset[1], label, color=color, fontsize=12)

def setup(ax, xlim, ylim, title=""):
    "Equal axes, light grid, x- and y-axis lines."
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect("equal")
    ax.axhline(0, color="#bbbbbb", lw=1, zorder=0); ax.axvline(0, color="#bbbbbb", lw=1, zorder=0)
    ax.grid(alpha=0.25); ax.set_title(title)

def angle_deg(u, v):
    "Angle between two vectors in degrees."
    c = u @ v / (np.linalg.norm(u) * np.linalg.norm(v))
    return np.degrees(np.arccos(np.clip(c, -1, 1)))

def direction_deg(v):
    "Direction of a 2-D vector, measured counter-clockwise from the x-axis."
    return np.degrees(np.arctan2(v[1], v[0]))

def unit_circle(n=300):
    t = np.linspace(0, 2*np.pi, n)
    return np.vstack([np.cos(t), np.sin(t)])

### <span style="color:#1f4e79">1. Matrix–Vector Multiplication $Ax = b$</span>

* An $m \times n$ matrix $A$ acts on an $n \times 1$ vector $x$ and produces an $m \times 1$ vector $b$.
* Rule: number of **columns** of $A$ = number of **rows** of $x$.

#### <span style="color:#455a64">Example</span>

$$
A = \begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix}, \quad x = \begin{bmatrix} 5 \\ 2 \end{bmatrix}
\qquad\Longrightarrow\qquad
Ax = \begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix} \begin{bmatrix} 5 \\ 2 \end{bmatrix} = b
$$

#### <span style="color:#455a64">Method 1: the row perspective (dot products)</span>

* Each entry of $b$ = dot product of one **row** of $A$ with $x$.

* First entry of $b$:
$$(2 \times 5) + (3 \times 2) = 10 + 6 = 16$$

* Second entry of $b$:
$$(1 \times 5) + (4 \times 2) = 5 + 8 = 13$$

$$
b = \begin{bmatrix} 16 \\ 13 \end{bmatrix}
$$

#### <span style="color:#455a64">Method 2: the column perspective (linear combination)</span>

* $Ax$ = combination of the **columns** of $A$, weighted by the entries of $x$.

$$
Ax = 5 \begin{bmatrix} 2 \\ 1 \end{bmatrix} + 2 \begin{bmatrix} 3 \\ 4 \end{bmatrix}
= \begin{bmatrix} 10 \\ 5 \end{bmatrix} + \begin{bmatrix} 6 \\ 8 \end{bmatrix} = \begin{bmatrix} 16 \\ 13 \end{bmatrix}
$$

* Both views give the same answer:

$$
\begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix} \begin{bmatrix} 5 \\ 2 \end{bmatrix} = \begin{bmatrix} 16 \\ 13 \end{bmatrix}
$$

* $x$ values are like **weights**: "take 5 of column 1 and 2 of column 2".

In [ ]:
A = np.array([[2.0, 3.0],
              [1.0, 4.0]])
x = np.array([5.0, 2.0])

b_row    = np.array([A[0, :] @ x, A[1, :] @ x])     # row view: dot products
b_column = x[0] * A[:, 0] + x[1] * A[:, 1]          # column view: combination of columns

print("row view    b =", b_row)
print("column view b =", b_column)
print("A @ x         =", A @ x)

row view    b = [16. 13.]
column view b = [16. 13.]
A @ x         = [16. 13.]


#### <span style="color:#455a64">Visualization: building $b$ from the columns of A</span>

In [ ]:
def column_picture(x1=5.0, x2=2.0):
    A = np.array([[2.0, 3.0], [1.0, 4.0]])
    v1, v2 = A[:, 0], A[:, 1]
    b = x1*v1 + x2*v2

    fig, ax = plt.subplots(figsize=(7, 6.5))
    for k in range(-4, 9):                                   # skewed grid of the columns
        ax.plot(*np.array([k*v1 - 4*v2, k*v1 + 8*v2]).T, color="#dddddd", lw=0.8, zorder=0)
        ax.plot(*np.array([k*v2 - 4*v1, k*v2 + 8*v1]).T, color="#dddddd", lw=0.8, zorder=0)
    arrow(ax, x1*v1, color=BLUE, lw=2, ls="--")
    arrow(ax, x2*v2, tail=x1*v1, color=ORANGE, lw=2, ls="--")
    arrow(ax, v1, color=BLUE, lw=3.5, label="$v_1$")
    arrow(ax, v2, color=ORANGE, lw=3.5, label="$v_2$")
    arrow(ax, b, color=RED, lw=3, label=f"$b$ = ({b[0]:.1f}, {b[1]:.1f})")
    setup(ax, (-12, 30), (-12, 30), rf"$b = {x1:g}\,v_1 + {x2:g}\,v_2$  (column picture)")
    plt.show()

interact(column_picture,
         x1=FloatSlider(value=5, min=-3, max=6, step=0.5, description="x1"),
         x2=FloatSlider(value=2, min=-3, max=6, step=0.5, description="x2"));

interactive(children=(FloatSlider(value=5.0, description='x1', max=6.0, min=-3.0, step=0.5), FloatSlider(value…

* the columns $v_1 = (2,1)$ and $v_2 = (3,4)$
* Dashed $x_1 v_1$ + $x_2 v_2$ = $b = Ax$
* Gray grid: the **coordinate grid made by the columns**
* It is skewed because $v_1, v_2$ are not perpendicular

### <span style="color:#1f4e79">2. Are the Columns of $A$ Orthogonal?</span>

* Two vectors are **orthogonal** (perpendicular) when their dot product is zero:

$$v_1 \cdot v_2 = 0$$

* Columns of $A = \begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix}$:

$$v_1 = \begin{bmatrix} 2 \\ 1 \end{bmatrix}, \quad v_2 = \begin{bmatrix} 3 \\ 4 \end{bmatrix}$$

* Dot product:

$$v_1 \cdot v_2 = (2 \times 3) + (1 \times 4) = 6 + 4 = 10 \neq 0$$

* Angle between them:

$$\cos \theta = \frac{v_1\cdot v_2}{\|v_1\|\,\|v_2\|} = \frac{10}{\sqrt{5}\cdot 5} = \frac{2}{\sqrt{5}} \approx 0.894 \quad\Longrightarrow\quad \theta \approx 26.6^\circ$$

* **Conclusion:** the columns of $A$ are **not** orthogonal.

In [ ]:
v1, v2 = A[:, 0], A[:, 1]
print("v1 . v2 =", v1 @ v2)
print("cos(theta) =", (v1 @ v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))
print("theta =", angle_deg(v1, v2), "degrees")

v1 . v2 = 10.0
cos(theta) = 0.8944271909999159
theta = 26.565051177077994 degrees


### <span style="color:#1f4e79">3. Consider a Matrix with Orthogonal Columns</span>

#### <span style="color:#455a64">Example</span>

* Columns $v_1 = \begin{bmatrix} 2 \\ 1 \end{bmatrix}$ and $v_2 = \begin{bmatrix} -1 \\ 2 \end{bmatrix}$; same $x$ as before.

$$
A = \begin{bmatrix} 2 & -1 \\ 1 & 2 \end{bmatrix}, \quad x = \begin{bmatrix} 5 \\ 2 \end{bmatrix}
$$

#### <span style="color:#455a64">Orthogonality</span>

$$
v_1 \cdot v_2 = (2 \times -1) + (1 \times 2) = -2 + 2 = 0
$$

* Dot product $= 0$ ⟹ angle $= 90^\circ$ ⟹ the columns are **orthogonal**.

#### <span style="color:#455a64">Column perspective</span>

$$
b = Ax = x_1 v_1 + x_2 v_2 = 5 \begin{bmatrix} 2 \\ 1 \end{bmatrix} + 2 \begin{bmatrix} -1 \\ 2 \end{bmatrix}
= \begin{bmatrix} 10 \\ 5 \end{bmatrix} + \begin{bmatrix} -2 \\ 4 \end{bmatrix} = \begin{bmatrix} 8 \\ 9 \end{bmatrix}
$$

#### <span style="color:#455a64">Why orthogonal columns matter</span>

* $v_1$ and $v_2$ form a **perpendicular coordinate grid**.
* Given $b$, recover $x$ **without** elimination or a matrix inverse: one projection per coordinate.

$$
x_1 = \frac{b \cdot v_1}{\|v_1\|^2} = \frac{(8)(2) + (9)(1)}{5} = \frac{25}{5} = 5,
\qquad
x_2 = \frac{b \cdot v_2}{\|v_2\|^2} = \frac{(8)(-1) + (9)(2)}{5} = \frac{10}{5} = 2
$$

* Each coordinate is **decoupled** from the other: there is no geometric overlap between the columns.

In [ ]:
A_orth = np.array([[2.0, -1.0],
                   [1.0,  2.0]])
v1, v2 = A_orth[:, 0], A_orth[:, 1]
x = np.array([5.0, 2.0])
b = A_orth @ x

print("v1 . v2 =", v1 @ v2)
print("b = A x =", b)
print("x1 = b.v1/|v1|^2 =", (b @ v1) / (v1 @ v1))
print("x2 = b.v2/|v2|^2 =", (b @ v2) / (v2 @ v2))

v1 . v2 = 0.0
b = A x = [8. 9.]
x1 = b.v1/|v1|^2 = 5.0
x2 = b.v2/|v2|^2 = 2.0


#### <span style="color:#455a64">Visualization: coordinates by projection</span>

In [ ]:
MATRICES = {
    "orthogonal   [[2,-1],[1,2]]":       np.array([[2.0, -1.0], [1.0, 2.0]]),
    "non-orthogonal [[2,3],[1,4]]":      np.array([[2.0, 3.0], [1.0, 4.0]]),
    "orthonormal  [[3/5,-4/5],[4/5,3/5]]": np.array([[0.6, -0.8], [0.8, 0.6]]),
}

def projection_picture(matrix="orthogonal   [[2,-1],[1,2]]", x1=5.0, x2=2.0, show_circle=False):
    A = MATRICES[matrix]
    v1, v2 = A[:, 0], A[:, 1]
    b = x1*v1 + x2*v2
    c1, c2 = (b @ v1)/(v1 @ v1), (b @ v2)/(v2 @ v2)          # projection formula

    # fixed window for each matrix (all slider values fit)
    corners = np.array([a*v1 + c*v2 for a in (-3, 6) for c in (-3, 6)] + [[0, 0]])
    lo, hi = corners.min(axis=0) - 1.5, corners.max(axis=0) + 1.5
    L = max(hi - lo) / 2; mid = (hi + lo) / 2

    fig, ax = plt.subplots(figsize=(7, 6.5))
    for k in range(-4, 9):                                   # grid of the columns
        ax.plot(*np.array([k*v1 - 4*v2, k*v1 + 8*v2]).T, color="#e3e3e3", lw=0.8, zorder=0)
        ax.plot(*np.array([k*v2 - 4*v1, k*v2 + 8*v1]).T, color="#e3e3e3", lw=0.8, zorder=0)
    for v, c, col in [(v1, c1, BLUE), (v2, c2, ORANGE)]:
        ax.plot(*np.array([-8*v, 10*v]).T, color=col, lw=0.8, alpha=0.4)   # line of the column
        ax.plot(*np.array([b, c*v]).T, color=col, ls=":", lw=1.8)          # perpendicular drop
        ax.plot(*(c*v), "o", color=col, ms=6)
    arrow(ax, x1*v1, color=BLUE, lw=2, ls="--")
    arrow(ax, x2*v2, tail=x1*v1, color=ORANGE, lw=2, ls="--")
    arrow(ax, v1, color=BLUE, lw=3.5, label="$v_1$")
    arrow(ax, v2, color=ORANGE, lw=3.5, label="$v_2$")
    arrow(ax, b, color=RED, lw=3, label="$b$")
    if show_circle:
        r = np.hypot(x1, x2); C = unit_circle()
        ax.plot(r*C[0], r*C[1], color=GRAY, ls="--", lw=1, label=rf"circle of radius $\|x\|$ = {r:.2f}")
        ax.legend(loc="lower right", fontsize=9)
    setup(ax, (mid[0]-L, mid[0]+L), (mid[1]-L, mid[1]+L),
          f"true x = ({x1:g}, {x2:g})    projection formula gives ({c1:.2f}, {c2:.2f})")
    plt.show()

interact(projection_picture,
         matrix=Dropdown(options=list(MATRICES), value="orthogonal   [[2,-1],[1,2]]", description="matrix"),
         x1=FloatSlider(value=5, min=-3, max=6, step=0.5, description="x1"),
         x2=FloatSlider(value=2, min=-3, max=6, step=0.5, description="x2"),
         show_circle=False);

interactive(children=(Dropdown(description='matrix', options=('orthogonal   [[2,-1],[1,2]]', 'non-orthogonal […

* The title compares the true $x$ with the projection formula $\dfrac{b\cdot v_i}{\|v_i\|^2}$.
* **Orthogonal** columns: projection gives $x$ exactly.
* **Non-orthogonal** columns: projection gives the **wrong** numbers. Try it!
* The dashed circle has radius $\|x\|$

### <span style="color:#1f4e79">4. Matrix with Orthonormal Columns</span>

* **Orthonormal** = orthogonal **and** every vector has length (norm) exactly $1$.

$$
A = \begin{bmatrix} 3/5 & -4/5 \\ 4/5 & 3/5 \end{bmatrix}, \quad x = \begin{bmatrix} 5 \\ 2 \end{bmatrix},
\qquad
q_1 = \begin{bmatrix} 3/5 \\ 4/5 \end{bmatrix}, \quad q_2 = \begin{bmatrix} -4/5 \\ 3/5 \end{bmatrix}
$$

#### <span style="color:#455a64">Orthonormality</span>

* Orthogonal (dot product zero):
$$
q_1 \cdot q_2 = \left(\tfrac{3}{5}\right)\left(-\tfrac{4}{5}\right) + \left(\tfrac{4}{5}\right)\left(\tfrac{3}{5}\right) = -\tfrac{12}{25} + \tfrac{12}{25} = 0
$$

* Unit length:
$$
\|q_1\|^2 = \tfrac{9}{25} + \tfrac{16}{25} = 1, \qquad \|q_2\|^2 = \tfrac{16}{25} + \tfrac{9}{25} = 1
$$

#### <span style="color:#455a64">$Ax = b$</span>

$$
b = 5 q_1 + 2 q_2 = \begin{bmatrix} 3 \\ 4 \end{bmatrix} + \begin{bmatrix} -1.6 \\ 1.2 \end{bmatrix}
= \begin{bmatrix} 1.4 \\ 5.2 \end{bmatrix} = \begin{bmatrix} 7/5 \\ 26/5 \end{bmatrix}
$$

#### <span style="color:#455a64">The power of orthonormality</span>

* Orthogonal columns: $\;x_i = \dfrac{b \cdot v_i}{\|v_i\|^2}$
* Orthonormal columns ($\|q_i\| = 1$): the denominator disappears:

$$
x_i = b \cdot q_i
$$

* Check:
$$
x_1 = \tfrac{7}{5}\cdot\tfrac{3}{5} + \tfrac{26}{5}\cdot\tfrac{4}{5} = \tfrac{21 + 104}{25} = 5,
\qquad
x_2 = \tfrac{7}{5}\cdot\left(-\tfrac{4}{5}\right) + \tfrac{26}{5}\cdot\tfrac{3}{5} = \tfrac{-28 + 78}{25} = 2
$$

#### <span style="color:#455a64">Matrix form</span>

$$
A^T A = I \qquad\Longrightarrow\qquad Ax = b \;\Longrightarrow\; x = A^T b
$$

* No elimination, no inverse: **the inverse is the transpose**.

In [ ]:
Q = np.array([[3/5, -4/5],
              [4/5,  3/5]])
x = np.array([5.0, 2.0])
b = Q @ x

print("q1 . q2 =", round(Q[:, 0] @ Q[:, 1], 12), "   |q1| =", np.linalg.norm(Q[:, 0]), "   |q2| =", np.linalg.norm(Q[:, 1]))
print("b = Q x  =", b)
print("x = Q^T b =", Q.T @ b)
print("Q^T Q =\n", Q.T @ Q)

q1 . q2 = 0.0    |q1| = 1.0    |q2| = 1.0
b = Q x  = [1.4 5.2]
x = Q^T b = [5. 2.]
Q^T Q =
 [[1. 0.]
 [0. 1.]]


#### <span style="color:#455a64">Visualization</span>

In [ ]:
interact(projection_picture,
         matrix=Dropdown(options=list(MATRICES), value="orthonormal  [[3/5,-4/5],[4/5,3/5]]", description="matrix"),
         x1=FloatSlider(value=5, min=-3, max=6, step=0.5, description="x1"),
         x2=FloatSlider(value=2, min=-3, max=6, step=0.5, description="x2"),
         show_circle=True);

interactive(children=(Dropdown(description='matrix', index=2, options=('orthogonal   [[2,-1],[1,2]]', 'non-ort…

* Same figure as in Section 3, now with the **orthonormal** matrix and $x = (5,2)$.
* The grid is made of **unit squares**, rotated by $53.13^\circ$.
* The dashed circle has radius $\|x\|$. Notice: $b$ ends **on** the circle.

#### <span style="color:#455a64">Is $A^TA = I$ the same as $A^{-1} = A^T$?</span>
* Two claims for a matrix with orthonormal columns:
  1. $A^T A = I$
  2. $A^{-1} = A^T$
* Claim 1 is always true. Claim 2 needs a **square** matrix.

#### <span style="color:#455a64">Claim 1: $A^TA = I$</span>

* $A$ is $m \times n$ with orthonormal columns $q_1, \dots, q_n \in \mathbb{R}^m$:

$$
A = \begin{bmatrix} | & | & & | \\ q_1 & q_2 & \dots & q_n \\ | & | & & | \end{bmatrix},
\qquad
A^T = \begin{bmatrix} \text{---} & q_1^T & \text{---} \\ \text{---} & q_2^T & \text{---} \\ & \vdots & \\ \text{---} & q_n^T & \text{---} \end{bmatrix}
$$

* Entry $(i,j)$ of $A^TA$ = row $i$ of $A^T$ times column $j$ of $A$:

$$
(A^T A)_{ij} = q_i^T q_j = q_i \cdot q_j
$$

* Orthonormality:
  * $i = j$ (diagonal): $\;q_i \cdot q_i = \|q_i\|^2 = 1$
  * $i \neq j$ (off-diagonal): $\;q_i \cdot q_j = 0$

$$
A^T A = \begin{bmatrix} q_1 \cdot q_1 & q_1 \cdot q_2 & \dots & q_1 \cdot q_n \\ q_2 \cdot q_1 & q_2 \cdot q_2 & \dots & q_2 \cdot q_n \\ \vdots & \vdots & \ddots & \vdots \\ q_n \cdot q_1 & q_n \cdot q_2 & \dots & q_n \cdot q_n \end{bmatrix}
= \begin{bmatrix} 1 & 0 & \dots & 0 \\ 0 & 1 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & 1 \end{bmatrix} = I_n
$$

* Holds for **any** matrix with orthonormal columns: square or tall ($m > n$).

#### <span style="color:#455a64">Claim 2: $A^{-1} = A^T$</span>

* A true (two-sided) inverse needs:
  1. $A$ **square** ($n \times n$)
  2. $A^{-1}A = I_n$ **and** $AA^{-1} = I_n$

**Case A — square $A$**

* $A^TA = I_n$ ⟹ $A^T$ is a left inverse.
* For square matrices a left inverse is also a right inverse: $AA^T = I_n$.
* Hence $\;A^{-1} = A^T$.
* Name: **orthogonal matrix** (complex case: **unitary** matrix).

**Case B — rectangular $A$ ($m > n$)**

$$
A = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \end{bmatrix}
$$

$$
A^T A = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \end{bmatrix} \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \end{bmatrix} = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} = I_2
$$

$$
A A^T = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \end{bmatrix} \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \end{bmatrix} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 0 \end{bmatrix} \neq I_3
$$

* $AA^T$ is a **projection** matrix, not the identity.
* $A^T$ is only a **left inverse** (it is also the Moore–Penrose pseudoinverse $A^+$).

#### <span style="color:#455a64">Verdict</span>

* $A^TA = I$: **always** true for orthonormal columns.
* $A^{-1} = A^T$: true **only if $A$ is square**.

In [ ]:
A_tall = np.array([[1.0, 0.0],
                   [0.0, 1.0],
                   [0.0, 0.0]])
print("A^T A =\n", A_tall.T @ A_tall)
print("A A^T =\n", A_tall @ A_tall.T)

print("\nsquare orthonormal Q:  Q Q^T =\n", Q @ Q.T)

A^T A =
 [[1. 0.]
 [0. 1.]]
A A^T =
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]]

square orthonormal Q:  Q Q^T =
 [[1. 0.]
 [0. 1.]]


#### <span style="color:#455a64">Visualization: $AA^T$ is a projection</span>


In [ ]:
PLANES = {
    "xy-plane: q1=(1,0,0), q2=(0,1,0)": np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]]),
    "tilted:   q1=(1,0,0), q2=(0,1,1)/sqrt2": np.array([[1.0, 0.0], [0.0, 1/np.sqrt(2)], [0.0, 1/np.sqrt(2)]]),
}

def projection_3d(plane="xy-plane: q1=(1,0,0), q2=(0,1,0)", v1=1.0, v2=1.5, v3=2.0):
    A = PLANES[plane]
    v = np.array([v1, v2, v3])
    Pv = A @ (A.T @ v)                        # A A^T v

    fig = plt.figure(figsize=(7.5, 6.5))
    ax = fig.add_subplot(111, projection="3d")
    s, t = np.meshgrid(np.linspace(-2.5, 2.5, 2), np.linspace(-2.5, 2.5, 2))
    P = A[:, [0]][:, :, None]*s + A[:, [1]][:, :, None]*t
    ax.plot_surface(P[0], P[1], P[2], color=BLUE, alpha=0.15, edgecolor="none")
    for q, name in [(A[:, 0], "$q_1$"), (A[:, 1], "$q_2$")]:
        ax.quiver(0, 0, 0, *q, color=BLUE, lw=2.5, arrow_length_ratio=0.15)
        ax.text(*(1.1*q), name, color=BLUE)
    ax.quiver(0, 0, 0, *v, color=RED, lw=2.5, arrow_length_ratio=0.08)
    ax.quiver(0, 0, 0, *Pv, color=GREEN, lw=2.5, arrow_length_ratio=0.1)
    ax.plot(*np.array([v, Pv]).T, color=GRAY, ls=":", lw=2)
    ax.text(*v, "  $v$", color=RED); ax.text(*Pv, "  $AA^Tv$", color=GREEN)
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_zlim(-2.5, 2.5); ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title(f"v = {np.round(v, 2)}    A A^T v = {np.round(Pv, 2)}")
    plt.show()

interact(projection_3d,
         plane=Dropdown(options=list(PLANES), description="plane"),
         v1=FloatSlider(value=1.0, min=-2, max=2, step=0.25, description="v1", continuous_update=False),
         v2=FloatSlider(value=1.5, min=-2, max=2, step=0.25, description="v2", continuous_update=False),
         v3=FloatSlider(value=2.0, min=-2, max=2, step=0.25, description="v3", continuous_update=False));

interactive(children=(Dropdown(description='plane', options=('xy-plane: q1=(1,0,0), q2=(0,1,0)', 'tilted:   q1…

* The two orthonormal columns span a **plane** in $\mathbb{R}^3$ (blue).
* $AA^Tv$ = the **shadow** of $v$ on that plane (green); the dotted drop is perpendicular to the plane.
* $AA^Tv = v$ only if $v$ already lies in the plane. So $AA^T \neq I$.
* Choose the plane and move $v$.

### <span style="color:#1f4e79">5. Orthonormal Columns Preserve Lengths and Angles</span>

* Question: when the orthonormal $A$ acted on $x = (5,2)$, did the **length** change?

#### <span style="color:#455a64">Compare lengths</span>

$$
\|x\|^2 = 5^2 + 2^2 = 29 \quad\Longrightarrow\quad \|x\| = \sqrt{29} \approx 5.385
$$

$$
b = \begin{bmatrix} 7/5 \\ 26/5 \end{bmatrix}: \qquad
\|b\|^2 = \tfrac{49}{25} + \tfrac{676}{25} = \tfrac{725}{25} = 29 \quad\Longrightarrow\quad \|b\| = \sqrt{29}
$$

* **The length did not change.**

#### <span style="color:#455a64">Proof for any $x$</span>

$$
\|Ax\|^2 = (Ax)^T (Ax) = x^T (A^T A)\, x = x^T I\, x = x^T x = \|x\|^2
$$

* So $\;\|Ax\| = \|x\|$: the transformation cannot stretch or compress any vector.

#### <span style="color:#455a64">Angles (dot products) are preserved</span>

$$
\cos \theta = \frac{x \cdot y}{\|x\|\, \|y\|}
$$

* Take $x = \begin{bmatrix} 5 \\ 2 \end{bmatrix}$ and $y = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$.

**Before:**
$$
x \cdot y = 5, \quad \|x\| = \sqrt{29}, \quad \|y\| = 1 \quad\Longrightarrow\quad \cos\theta = \frac{5}{\sqrt{29}} \approx 0.928, \quad \theta \approx 21.8^\circ
$$

**After:**
$$
Ay = \begin{bmatrix} 3/5 \\ 4/5 \end{bmatrix}, \qquad
(Ax) \cdot (Ay) = \tfrac{7}{5}\cdot\tfrac{3}{5} + \tfrac{26}{5}\cdot\tfrac{4}{5} = \tfrac{125}{25} = 5
\quad\Longrightarrow\quad \cos\theta = \frac{5}{\sqrt{29}}
$$

* Same dot product, same lengths ⟹ **same angle**.
* In general: $(Ax)\cdot(Ay) = x^TA^TAy = x\cdot y$.

#### <span style="color:#455a64">Geometric meaning: rigid motion</span>

* Lengths and angles preserved ⟹ $A$ does not warp space (an **isometry**).
* In 2-D every square orthonormal matrix is one of two types:

1. **Rotation** by angle $\theta$:
$$R = \begin{bmatrix} \cos \theta & -\sin \theta \\ \sin \theta & \cos \theta \end{bmatrix}, \qquad \det R = +1$$

2. **Reflection** across the line at angle $\theta/2$:
$$F = \begin{bmatrix} \cos \theta & \sin \theta \\ \sin \theta & -\cos \theta \end{bmatrix}, \qquad \det F = -1$$

* Our example: $\cos\theta = 3/5$, $\sin\theta = 4/5$ ⟹ rotation by $\theta \approx 53.13^\circ$ counter-clockwise.

In [ ]:
x, y = np.array([5.0, 2.0]), np.array([1.0, 0.0])
print("|x| =", np.linalg.norm(x), "  |Qx| =", np.linalg.norm(Q @ x))
print("x.y =", x @ y, "  (Qx).(Qy) =", (Q @ x) @ (Q @ y))
print("angle(x, y) =", angle_deg(x, y), "  angle(Qx, Qy) =", angle_deg(Q @ x, Q @ y))
print("rotation angle of Q =", np.degrees(np.arctan2(Q[1, 0], Q[0, 0])), "degrees;  det Q =", np.linalg.det(Q))

|x| = 5.385164807134504   |Qx| = 5.385164807134505
x.y = 5.0   (Qx).(Qy) = 5.0
angle(x, y) = 21.801409486351798   angle(Qx, Qy) = 21.801409486351833
rotation angle of Q = 53.13010235415599 degrees;  det Q = 1.0


#### <span style="color:#455a64">Visualization: rotation and reflection</span>

In [ ]:
F_SHAPE = np.array([[0, 0], [0, 2], [1.2, 2], [1.2, 1.7], [0.3, 1.7], [0.3, 1.15],
                    [0.95, 1.15], [0.95, 0.85], [0.3, 0.85], [0.3, 0], [0, 0]]).T * 1.5 + np.array([[1.0], [1.0]])

def rigid_motion(theta=53.13, kind="rotation"):
    t = np.radians(theta)
    if kind == "rotation":
        Qm = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
    else:
        Qm = np.array([[np.cos(t), np.sin(t)], [np.sin(t), -np.cos(t)]])
    x, y = np.array([5.0, 2.0]), 2*np.array([1.0, 0.0])
    Qx, Qy = Qm @ x, Qm @ y

    fig, ax = plt.subplots(figsize=(5.8, 5.5))
    r = np.linalg.norm(x); C = unit_circle()
    ax.plot(r*C[0], r*C[1], color=GRAY, ls="--", lw=1)
    ax.fill(*F_SHAPE, color=GRAY, alpha=0.3); ax.fill(*(Qm @ F_SHAPE), color=GREEN, alpha=0.45)
    arrow(ax, x, color=GRAY, lw=2.5, label="$x$"); arrow(ax, y, color=GRAY, lw=2.5, label="$y$")
    arrow(ax, Qx, color=RED, lw=3, label="$Qx$"); arrow(ax, Qy, color=BLUE, lw=3, label="$Qy$")
    if kind == "reflection":
        d = np.array([np.cos(t/2), np.sin(t/2)])
        ax.plot(*np.array([-7*d, 7*d]).T, color="k", ls="-.", lw=1, label="mirror line")
        ax.legend(loc="lower left", fontsize=9)
    setup(ax, (-6.5, 6.5), (-6.5, 6.5),
          f"{kind}, det = {np.linalg.det(Qm):+.0f}\n"
          f"|x| = {np.linalg.norm(x):.3f} -> |Qx| = {np.linalg.norm(Qx):.3f};  "
          f"angle(x,y) = {angle_deg(x, y):.1f}° -> {angle_deg(Qx, Qy):.1f}°")
    plt.show()

interact(rigid_motion,
         theta=FloatSlider(value=53.13, min=0, max=360, step=1, description="theta (deg)"),
         kind=ToggleButtons(options=["rotation", "reflection"], description="type"));

interactive(children=(FloatSlider(value=53.13, description='theta (deg)', max=360.0, step=1.0), ToggleButtons(…

* Slider $\theta$; buttons: modify rotation or reflection matrices
* Testing a vector (length and dot product): $x = (5,2)$ and $y = (1,0)$ under rotation and reflection
* Or any general letter, say, "F"
* The title shows lengths and angles **before → after**: they never change.
* Reflection: the "F" is mirrored (orientation flips, $\det = -1$).

### <span style="color:#1f4e79">6. Orthogonal or Unitary?</span>

* **Orthogonal transformation:** a linear map that preserves dot products ⟹ lengths and angles.
* Applying our real $A$ with orthonormal columns to $x$ **is** an orthogonal transformation.
* **Unitary transformation:** the same idea on a **complex** vector space; it preserves the complex inner product $x^*y = \sum_i \overline{x_i}\, y_i$.
* $U^*$ = conjugate transpose (physics notation: $U^\dagger$).

| | Real numbers | Complex numbers |
|---|---|---|
| matrix name | **orthogonal** matrix | **unitary** matrix |
| defining property | $Q^TQ = I$ | $U^*U = I$ |
| inverse | $Q^{-1} = Q^T$ | $U^{-1} = U^*$ |
| transformation | orthogonal transformation | unitary transformation |

* Our $A$ has only real entries ($3/5, -4/5, 4/5$) ⟹ **orthogonal matrix**
* An orthogonal matrix is a unitary matrix that happens to be real.

#### <span style="color:#455a64">Columns of an orthogonal matrix are orthonormal</span>

* Definition: square $Q$ with $Q^TQ = I$.
* Entry $(i,j)$ of $Q^TQ$ is $q_i\cdot q_j$:
  * diagonal entries $= 1$ ⟹ unit length
  * off-diagonal entries $= 0$ ⟹ mutually perpendicular
* So the columns of an orthogonal matrix are **always orthonormal** (and so are its rows, since $QQ^T = I$ too).
* Note the naming trap: "orthogonal matrix" means ortho**normal** columns.

In [ ]:
U = np.array([[1, 1],
              [1j, -1j]]) / np.sqrt(2)          # a complex unitary matrix
z = np.array([1 + 2j, 3 - 1j])

print("U^* U =\n", U.conj().T @ U)
print("|z| =", np.linalg.norm(z), "   |Uz| =", np.linalg.norm(U @ z))
print("rows of Q are orthonormal too:  Q Q^T =\n", Q @ Q.T)

U^* U =
 [[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]
|z| = 3.872983346207417    |Uz| = 3.8729833462074166
rows of Q are orthonormal too:  Q Q^T =
 [[1. 0.]
 [0. 1.]]


### <span style="color:#1f4e79">7. A General Matrix Stretches **and** Rotates</span>

* Back to the first example: $A = \begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix}$, $\;x = \begin{bmatrix} 5 \\ 2 \end{bmatrix}$, $\;b = \begin{bmatrix} 16 \\ 13 \end{bmatrix}$.
* Both $x$ and $b$ are drawn from the origin, so we can compare them directly.

* **Stretching:**
$$\|x\| = \sqrt{29} \approx 5.39 \quad\longrightarrow\quad \|b\| = \sqrt{425} \approx 20.62 \qquad (\text{factor} \approx 3.83)$$

* **Rotation:**
$$\text{direction of } x \approx 21.8^\circ \quad\longrightarrow\quad \text{direction of } b \approx 39.1^\circ \qquad (\approx 17.3^\circ \text{ counter-clockwise})$$

* Unlike an orthogonal matrix, $A$ changes **both** length and direction.

In [ ]:
A = np.array([[2.0, 3.0],
              [1.0, 4.0]])
x = np.array([5.0, 2.0])
b = A @ x
print("|x| =", np.linalg.norm(x), "   |b| =", np.linalg.norm(b), "   stretch factor =", np.linalg.norm(b)/np.linalg.norm(x))
print("direction of x =", direction_deg(x), "   direction of b =", direction_deg(b),
      "   rotation =", direction_deg(b) - direction_deg(x))

|x| = 5.385164807134504    |b| = 20.615528128088304    stretch factor = 3.8282074674438826
direction of x = 21.80140948635181    direction of b = 39.0938588862295    rotation = 17.292449399877686


#### <span style="color:#455a64">Different directions are stretched differently</span>

In [ ]:
def stretch_picture(phi=21.8):
    A = np.array([[2.0, 3.0], [1.0, 4.0]])
    sig = np.linalg.svd(A, compute_uv=False)
    C = unit_circle(400); AC = A @ C
    p = np.radians(phi); x = np.array([np.cos(p), np.sin(p)]); Ax = A @ x

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw=dict(width_ratios=[1, 1.2]))
    ax1.plot(*C, color=BLUE, lw=1.5, label="unit circle")
    ax1.plot(*AC, color=RED, lw=1.5, label="image: ellipse")
    arrow(ax1, x, color=BLUE, lw=3, label="$x$")
    arrow(ax1, Ax, color=RED, lw=3, label="$Ax$")
    ax1.legend(loc="lower right", fontsize=9)
    setup(ax1, (-6, 6), (-6, 6),
          f"|Ax| = {np.linalg.norm(Ax):.3f},  rotation = {direction_deg(Ax) - phi:+.1f}°")

    phis = np.linspace(0, 360, 721)
    ax2.plot(phis, np.linalg.norm(A @ np.vstack([np.cos(np.radians(phis)), np.sin(np.radians(phis))]), axis=0),
             color=RED, lw=2)
    ax2.axhline(sig[0], color=GRAY, ls="--"); ax2.axhline(sig[1], color=GRAY, ls="--")
    ax2.text(2, sig[0] + 0.15, rf"$\sigma_1$ = {sig[0]:.3f} (max stretch)")
    ax2.text(2, sig[1] + 0.15, rf"$\sigma_2$ = {sig[1]:.3f} (min stretch)")
    ax2.plot(phi, np.linalg.norm(Ax), "o", color=RED, ms=9)
    ax2.set_xlim(0, 360); ax2.set_ylim(0, 6.2); ax2.set_xticks(range(0, 361, 45))
    ax2.set_xlabel(r"direction of $x$, $\varphi$ (degrees)"); ax2.set_ylabel(r"stretch factor $\|Ax\|$")
    ax2.grid(alpha=0.3); ax2.set_title("stretch factor for every direction")
    plt.tight_layout(); plt.show()

interact(stretch_picture, phi=FloatSlider(value=21.8, min=0, max=360, step=0.5, description="phi (deg)"));

interactive(children=(FloatSlider(value=21.8, description='phi (deg)', max=360.0, step=0.5), Output()), _dom_c…

* Take **unit** vectors $x$ in every direction $\varphi$ (the unit circle).
* Their images $Ax$ trace an **ellipse**.
* Right panel: stretch factor $\|Ax\|$ as a function of the direction $\varphi$.
* The **largest** and **smallest** stretch factors are special numbers: $\sigma_1 \approx 5.398$ and $\sigma_2 \approx 0.926$.
* These are the **singular values** of $A$. They are the semi-axes of the ellipse.
* Start value: $\varphi = 21.8^\circ$, the direction of $x = (5,2)$.

### <span style="color:#1f4e79">8. Why the SVD?</span>

* The SVD reveals the structure of **any** matrix.
* Every linear transformation = three simple steps:

$$
A = U \Sigma V^T
$$

| Step | Matrix | What it does |
|---|---|---|
| 1 | $V^T$ (orthogonal) | **rotate** (or reflect) the input |
| 2 | $\Sigma$ (diagonal, $\sigma_i \ge 0$) | **stretch** along the coordinate axes |
| 3 | $U$ (orthogonal) | **rotate** (or reflect) to the output |

**Why it is useful**

* Works for **every** matrix: square or rectangular, invertible or not.
* Solves $Ax=b$ even when $A$ is not invertible (pseudoinverse).
* Data compression: keep only the large $\sigma_i$ (low-rank approximation).
* Diagnostics: rank, norms, condition number come straight from $\Sigma$.

#### <span style="color:#455a64">SVD of our example</span>

* $A = \begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix} = U \Sigma V^T$ with

$$
U \approx \begin{bmatrix} 0.655 & -0.755 \\ 0.755 & 0.655 \end{bmatrix}, \quad
\Sigma \approx \begin{bmatrix} 5.398 & 0 \\ 0 & 0.926 \end{bmatrix}, \quad
V^T \approx \begin{bmatrix} 0.383 & 0.924 \\ -0.924 & 0.383 \end{bmatrix}
$$

* Check: $\sigma_1\sigma_2 = 5.398 \times 0.926 \approx 5 = |\det A|$ (area scaling).
* **Signs:** NumPy may return $u_i$ and $v_i$ with flipped signs. Flipping **both** $u_i$ and $v_i$ keeps $A = U\Sigma V^T$.
* Below we choose the signs so that $U$ and $V$ are **rotations** ($\det = +1$), as in the hand values above.

In [ ]:
A = np.array([[2.0, 3.0],
              [1.0, 4.0]])
U, s, Vt = np.linalg.svd(A)
if np.linalg.det(Vt) < 0:          # flip u1 and v1 together -> U and V become rotations
    U[:, 0] *= -1
    Vt[0, :] *= -1
Sigma = np.diag(s)

print("U =\n", U, "\nSigma =\n", Sigma, "\nV^T =\n", Vt)
print("det U =", round(np.linalg.det(U), 6), "  det V =", round(np.linalg.det(Vt), 6))
print("U Sigma V^T =\n", U @ Sigma @ Vt)
print("sigma1 * sigma2 =", s.prod(), "   det A =", np.linalg.det(A))

U =
 [[ 0.6552 -0.7555]
 [ 0.7555  0.6552]] 
Sigma =
 [[5.3983 0.    ]
 [0.     0.9262]] 
V^T =
 [[ 0.3827  0.9239]
 [-0.9239  0.3827]]
det U = 1.0   det V = 1.0
U Sigma V^T =
 [[2. 3.]
 [1. 4.]]
sigma1 * sigma2 = 5.0    det A = 5.000000000000001


#### <span style="color:#455a64">Applying the three pieces one by one to $x = (5,2)$</span>

1. **First rotation** $x_1 = V^T x$: length unchanged; aligns $x$ with the stretching axes
$$x_1 \approx \begin{bmatrix} 3.761 \\ -3.854 \end{bmatrix}$$

2. **Stretch** $x_2 = \Sigma x_1$: pure stretching along the axes
$$x_2 \approx \begin{bmatrix} 5.398 \times 3.761 \\ 0.926 \times (-3.854) \end{bmatrix} = \begin{bmatrix} 20.304 \\ -3.570 \end{bmatrix}$$

3. **Final rotation** $b = U x_2$: length unchanged; final orientation
$$b \approx \begin{bmatrix} 16 \\ 13 \end{bmatrix} = Ax$$

In [ ]:
x  = np.array([5.0, 2.0])
x1 = Vt @ x
x2 = Sigma @ x1
x3 = U @ x2

for name, v in [("x", x), ("x1 = V^T x", x1), ("x2 = Sigma x1", x2), ("b = U x2", x3)]:
    print(f"{name:14s} = {v},   length = {np.linalg.norm(v):.4f}")
print("A x            =", A @ x)

x              = [5. 2.],   length = 5.3852
x1 = V^T x     = [ 3.7612 -3.854 ],   length = 5.3852
x2 = Sigma x1  = [20.3041 -3.5696],   length = 20.6155
b = U x2       = [16. 13.],   length = 20.6155
A x            = [16. 13.]


#### <span style="color:#455a64">Visualization: rotate → stretch → rotate</span>

In [ ]:
def rot(a):
    return np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])

aV = np.arctan2(Vt[1, 0], Vt[0, 0])       # V^T is a rotation by aV
aU = np.arctan2(U[1, 0], U[0, 0])         # U   is a rotation by aU

def partial_map(t):
    "Matrix after 'stage' t in [0, 3]."
    if t <= 1:
        return rot(t*aV)
    if t <= 2:
        return np.diag(1 + (t - 1)*(s - 1)) @ Vt
    return rot((t - 2)*aU) @ Sigma @ Vt

def svd_stages(stage=0.0):
    M_t = partial_map(stage)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

    x = np.array([5.0, 2.0])
    names = ["$x$", "$x_1 = V^Tx$", r"$x_2 = \Sigma x_1$", "$b = Ux_2$"]
    fixed = [x, Vt @ x, Sigma @ Vt @ x, A @ x]
    for k, (v, nm, col) in enumerate(zip(fixed, names, [BLUE, GREEN, ORANGE, RED])):
        arrow(ax1, v, color=col, lw=1.5, alpha=0.35, label=nm)
    arrow(ax1, M_t @ x, color="k", lw=3.5)
    setup(ax1, (-8, 24), (-8, 24), f"stage = {stage:.2f}:  current vector = {np.round(M_t @ x, 2)}")

    C = unit_circle(); MC = M_t @ C
    ax2.fill(*MC, color=BLUE, alpha=0.12); ax2.plot(*MC, color=BLUE, lw=2)
    arrow(ax2, M_t @ Vt[0], color=RED, lw=3, label="image of $v_1$")
    arrow(ax2, M_t @ Vt[1], color=GREEN, lw=3, label="image of $v_2$")
    setup(ax2, (-6, 6), (-6, 6), "the unit circle under the same steps")
    plt.tight_layout(); plt.show()

interact(svd_stages, stage=FloatSlider(value=0, min=0, max=3, step=0.05, description="stage"));

interactive(children=(FloatSlider(value=0.0, description='stage', max=3.0, step=0.05), Output()), _dom_classes…

* Slider **stage** from 0 to 3 plays the three steps continuously:
  * 0 → 1: rotate by $V^T$
  * 1 → 2: stretch by $\Sigma$
  * 2 → 3: rotate by $U$
* Left: the actual vectors $x \to x_1 \to x_2 \to b$ (current one bold).
* Right: what happens to the whole **unit circle**. Red and green: the directions $v_1, v_2$ being carried along.
  * After $V^T$: $v_1, v_2$ lie on the coordinate axes.
  * After $\Sigma$: circle → ellipse with semi-axes $\sigma_1, \sigma_2$.
  * After $U$: the ellipse turns into place; $v_1, v_2$ end as $\sigma_1u_1, \sigma_2u_2$.

#### <span style="color:#455a64">Where Do $U$, $\Sigma$, $V$ Come From?</span>

#### <span style="color:#455a64">The key relation $Av = \sigma u$</span>

$$
A v_i = \sigma_i u_i
$$

#### <span style="color:#455a64">By hand for our example</span>

$$
A^TA = \begin{bmatrix} 2 & 1 \\ 3 & 4 \end{bmatrix}\begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix} = \begin{bmatrix} 5 & 10 \\ 10 & 25 \end{bmatrix}
$$

* Characteristic equation: $\lambda^2 - 30\lambda + 25 = 0$ (trace $= 30$, determinant $= 25$):

$$
\lambda = 15 \pm \sqrt{200} \;\approx\; 29.142,\; 0.858
\qquad\Longrightarrow\qquad
\sigma_1 = \sqrt{29.142} \approx 5.398, \quad \sigma_2 = \sqrt{0.858} \approx 0.926
$$

* Eigenvector of $\lambda_1$: $\;v_1 \approx (0.383,\ 0.924)$ (direction $67.5^\circ$); $\;v_2 \perp v_1$.
* Then $u_1 = Av_1/\sigma_1 \approx (0.655,\ 0.755)$, $\;u_2 = Av_2/\sigma_2$.

---

* $A$ maps an input principal axis $v_i$ onto an output principal axis $u_i$, stretched by $\sigma_i$.
* In matrix form: $AV = U\Sigma$ ⟹ $A = U\Sigma V^T$.

#### <span style="color:#455a64">Step 1: $V$ and $\Sigma$ from $A^TA$</span>

$$
A^TA = (U\Sigma V^T)^T(U\Sigma V^T) = V\Sigma^T U^TU\Sigma V^T = V\Sigma^2V^T
$$

* Columns of $V$ = **eigenvectors** of $A^TA$.
* $\sigma_i = \sqrt{\lambda_i}$, where $\lambda_i$ = **eigenvalues** of $A^TA$.

#### <span style="color:#455a64">Step 2: $U$ from $u_i = \dfrac{Av_i}{\sigma_i}$</span>

---

In [ ]:
lam, V = np.linalg.eigh(A.T @ A)          # eigenvalues in ascending order
lam, V = lam[::-1], V[:, ::-1]            # sort descending
sig = np.sqrt(lam)
U_hand = (A @ V) / sig                    # u_i = A v_i / sigma_i

print("A^T A =\n", A.T @ A)
print("eigenvalues lambda =", lam, "  ->  sigma =", sig)
print("V (columns v1, v2) =\n", V)
print("U (columns u1, u2) =\n", U_hand)
print("compare np.linalg.svd: sigma =", np.linalg.svd(A, compute_uv=False))
print("A v_i = sigma_i u_i ?", np.allclose(A @ V, U_hand * sig))

A^T A =
 [[ 5. 10.]
 [10. 25.]]
eigenvalues lambda = [29.1421  0.8579]   ->  sigma = [5.3983 0.9262]
V (columns v1, v2) =
 [[ 0.3827 -0.9239]
 [ 0.9239  0.3827]]
U (columns u1, u2) =
 [[ 0.6552 -0.7555]
 [ 0.7555  0.6552]]
compare np.linalg.svd: sigma = [5.3983 0.9262]
A v_i = sigma_i u_i ? True


#### <span style="color:#455a64">Visualization: which perpendicular pair stays perpendicular?</span>

In [ ]:
def perpendicular_pair(phi=20.0):
    p = np.radians(phi)
    v, w = np.array([np.cos(p), np.sin(p)]), np.array([-np.sin(p), np.cos(p)])
    Av, Aw = A @ v, A @ w
    ang = angle_deg(Av, Aw)
    C = unit_circle()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5))
    ax1.plot(*C, color=BLUE, lw=1.5)
    for d in (Vt[0], Vt[1]):
        ax1.plot(*np.array([-1.3*d, 1.3*d]).T, color=GRAY, ls=":", lw=1)   # true v1, v2 directions
    arrow(ax1, v, color=RED, lw=3, label="$v$"); arrow(ax1, w, color=GREEN, lw=3, label=r"$v^\perp$")
    setup(ax1, (-1.6, 1.6), (-1.6, 1.6), f"input: angle(v, v_perp) = 90.0°   (phi = {phi:.1f}°)")

    ax2.plot(*(A @ C), color=RED, lw=1.5)
    arrow(ax2, Av, color=RED, lw=3, label="$Av$"); arrow(ax2, Aw, color=GREEN, lw=3, label=r"$Av^\perp$")
    status = "perpendicular: these are v1, v2!" if abs(ang - 90) < 0.6 else "not perpendicular"
    setup(ax2, (-6, 6), (-6, 6), f"output: angle(Av, Av_perp) = {ang:.1f}°  ({status})")
    plt.tight_layout(); plt.show()

interact(perpendicular_pair, phi=FloatSlider(value=20, min=0, max=180, step=0.5, description="phi (deg)"));

interactive(children=(FloatSlider(value=20.0, description='phi (deg)', max=180.0, step=0.5), Output()), _dom_c…

* Left: a perpendicular pair of unit vectors $v, v^\perp$; turn it with the slider.
* Right: their images $Av$ and $Av^\perp$ on the ellipse.
* In general $Av$ and $Av^\perp$ are **not** perpendicular.
* At $\varphi \approx 67.5^\circ$ (the direction of $v_1$) they **are**: $Av_1 = \sigma_1u_1 \perp Av_2 = \sigma_2u_2$.
* **The SVD finds the one pair of perpendicular input directions that stay perpendicular.**

---

* **Note:** computing the SVD through $A^TA$ is good for understanding.
* Library routines (`np.linalg.svd`, LAPACK) avoid forming $A^TA$: it squares the condition number and loses small singular values

---

### <span style="color:#1f4e79">9. QR Decomposition</span>

$$
A = QR
$$

* $Q$: orthogonal — a rigid, perpendicular **frame** (rotation / reflection).
* $R$: upper triangular — the **coordinates** of the columns of $A$ in that frame (a "shear").
* Geometry: builds a perfect perpendicular coordinate system from **skewed** columns.
* Importance:
  * fast, numerically stable **least squares**
  * the engine of the **QR algorithm** for eigenvalues

#### <span style="color:#455a64">QR for our example</span>

* To calculate the QR decomposition of $A = \begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix}$ manually, we use Gram–Schmidt.
* Let $a_1 = \begin{bmatrix} 2 \\ 1 \end{bmatrix}$ and $a_2 = \begin{bmatrix} 3 \\ 4 \end{bmatrix}$.

**Find the first orthonormal column ($q_1$)**

* Normalize $a_1$:
$$\|a_1\| = \sqrt{5}, \qquad q_1 = \frac{a_1}{\|a_1\|} = \begin{bmatrix} 2/\sqrt{5} \\ 1/\sqrt{5} \end{bmatrix}$$

**Find the second orthonormal column ($q_2$)**

* Subtract the projection of $a_2$ onto $q_1$.
* Overlap: $\;a_2 \cdot q_1 = \frac{6}{\sqrt{5}} + \frac{4}{\sqrt{5}} = 2\sqrt{5}$
* Perpendicular component:
$$w_2 = a_2 - (a_2 \cdot q_1)\,q_1 = \begin{bmatrix} 3 \\ 4 \end{bmatrix} - \begin{bmatrix} 4 \\ 2 \end{bmatrix} = \begin{bmatrix} -1 \\ 2 \end{bmatrix}$$
* Normalize $w_2$:
$$\|w_2\| = \sqrt{5}, \qquad q_2 = \begin{bmatrix} -1/\sqrt{5} \\ 2/\sqrt{5} \end{bmatrix}$$

$$Q = \begin{bmatrix} 2/\sqrt{5} & -1/\sqrt{5} \\ 1/\sqrt{5} & 2/\sqrt{5} \end{bmatrix}$$

* Note: $Q$ is the orthogonal matrix

**Construct the triangular matrix ($R$)**

* $r_{ij} = a_j \cdot q_i$:
  * $r_{11} = \|a_1\| = \sqrt{5}$
  * $r_{12} = a_2 \cdot q_1 = 2\sqrt{5}$
  * $r_{22} = \|w_2\| = \sqrt{5}$
  * $r_{21} = 0$

$$R = \begin{bmatrix} \sqrt{5} & 2\sqrt{5} \\ 0 & \sqrt{5} \end{bmatrix}$$

**Verification ($A = QR$)**

* Multiplying $QR$ recovers $A = \begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix}$.

In [ ]:
A = np.array([[2.0, 3.0],
              [1.0, 4.0]])
Q, R = np.linalg.qr(A)
D = np.diag(np.sign(np.diag(R)))      # NumPy may flip signs; make r_jj > 0 as in the hand calculation
Q, R = Q @ D, D @ R

print("Q * sqrt(5) =\n", Q * np.sqrt(5))
print("R / sqrt(5) =\n", R / np.sqrt(5))
print("Q^T Q =\n", Q.T @ Q)
print("Q R =\n", Q @ R)

Q * sqrt(5) =
 [[ 2. -1.]
 [ 1.  2.]]
R / sqrt(5) =
 [[1. 2.]
 [0. 1.]]
Q^T Q =
 [[1. 0.]
 [0. 1.]]
Q R =
 [[2. 3.]
 [1. 4.]]


#### <span style="color:#455a64">Visualization: Gram–Schmidt step by step</span>

In [ ]:
def gram_schmidt_steps(step=1):
    a1, a2 = A[:, 0], A[:, 1]
    q1 = a1 / np.linalg.norm(a1)
    r12 = a2 @ q1
    w2 = a2 - r12*q1
    q2 = w2 / np.linalg.norm(w2)

    fig, ax = plt.subplots(figsize=(6.5, 6))
    arrow(ax, a1, color=GRAY, lw=2.5, label="$a_1$")
    arrow(ax, a2, color=GRAY, lw=2.5, label="$a_2$")
    title = "skewed columns $a_1, a_2$"
    if step >= 2:
        arrow(ax, q1, color=BLUE, lw=3.5, label="$q_1$", offset=(0.1, -0.45))
        title = rf"$r_{{11}} = \|a_1\|$ = {np.linalg.norm(a1):.3f}"
    if step >= 3:
        arrow(ax, r12*q1, color=ORANGE, lw=2, ls="--", label=r"$(a_2\cdot q_1)q_1$", offset=(0.15, -0.5))
        arrow(ax, w2, tail=r12*q1, color=GREEN, lw=2, ls="--", label="$w_2$", offset=(-0.6, -0.4))
        arrow(ax, w2, color=GREEN, lw=2, alpha=0.5)
        title += rf",  $r_{{12}} = a_2\cdot q_1$ = {r12:.3f}"
    if step >= 4:
        arrow(ax, q2, color=BLUE, lw=3.5, label="$q_2$", offset=(-0.5, 0.1))
        title += rf",  $r_{{22}} = \|w_2\|$ = {np.linalg.norm(w2):.3f}"
    setup(ax, (-1.8, 6), (-0.8, 5), title)
    plt.show()

interact(gram_schmidt_steps, step=IntSlider(value=1, min=1, max=4, step=1, description="step"));

interactive(children=(IntSlider(value=1, description='step', max=4, min=1), Output()), _dom_classes=('widget-i…

* Slider **step**:
  1. the skewed columns $a_1, a_2$
  2. $q_1 = a_1/\|a_1\|$
  3. the projection $(a_2\cdot q_1)\,q_1$ and the perpendicular part $w_2$
  4. $q_2 = w_2/\|w_2\|$ and $a_2 = r_{12}q_1 + r_{22}q_2$
* The title shows the entries of $R$ found so far.

#### <span style="color:#455a64">Can QR be applied to any matrix?</span>

* Mathematically **yes**: every matrix has a QR decomposition.
* Rank-deficient columns (some column depends on the others):
  * Gram–Schmidt breaks down (division by zero).
  * Householder still gives $A = QR$, but the first columns of $Q$ need not span $\mathrm{range}(A)$.
  * Remedy: **QR with column pivoting**, $AP = QR$.
* For deciding the rank reliably, the SVD is the safest tool.

#### <span style="color:#455a64">Algorithms for QR</span>

| Algorithm | Idea | Typical use |
|---|---|---|
| **Householder reflections** | reflect a column onto an axis, zeroing all entries below the diagonal at once | standard for dense matrices (LAPACK) |
| **Givens rotations** | rotate in one 2-D plane to zero **one** entry | sparse or structured matrices |
| **Modified Gram–Schmidt** | subtract projections one at a time | iterative solvers (Arnoldi, GMRES) |

#### <span style="color:#455a64">How QR and SVD are connected</span>

* QR and SVD are deeply and mathematically tethered to one another

* **The $A^T A$ Bridge:**
  $$A^T A = (QR)^T (QR) = R^T Q^T Q R = R^T R$$
  Because SVD tells us $A^T A = V \Sigma^2 V^T$, this means **$R^T R = V \Sigma^2 V^T$**. $R$ is the **Cholesky factor** of $A^TA$ (with $r_{jj} > 0$): the same symmetric matrix that builds the SVD

* **Geometric tradeoff:** SVD uses two rotations ($U, V^T$) to isolate stretch into a pure diagonal ($\Sigma$). QR uses one rotation ($Q$), forcing stretch and shear to remain tangled in the upper triangular matrix ($R$)

**The Connection Between $R$ and $\Sigma, V$:**

* **$A$ and $R$ share the exact same singular values ($\Sigma$):** $Q$ is a rigid rotation, so $R$ must contain all the stretching power of $A$. SVD on $R$ yields the same $\Sigma$ and $V^T$ as SVD on $A$
  
* By setting $QR = U \Sigma V^T$ and multiplying by $Q^T$, we get:
$$R = (Q^T U) \Sigma V^T = W \Sigma V^T$$

* $R$ is literally an SVD in disguise. The right-side rotation ($V^T$) and core stretching ($\Sigma$) are identical to $A$, with a different left-side rotation ($W$)

In [ ]:
U_A, s_A, Vt_A = np.linalg.svd(A)
U_R, s_R, Vt_R = np.linalg.svd(R)
print("singular values of A =", s_A)
print("singular values of R =", s_R)
print("V^T from A =\n", Vt_A, "\nV^T from R (same up to signs) =\n", Vt_R)
print("A^T A =\n", A.T @ A, "\nR^T R =\n", R.T @ R)

W = Q.T @ U_A                                 # R = W Sigma V^T with W = Q^T U
print("W^T W =\n", W.T @ W)
print("W Sigma V^T =\n", W @ np.diag(s_A) @ Vt_A, "\nR =\n", R)

singular values of A = [5.3983 0.9262]
singular values of R = [5.3983 0.9262]
V^T from A =
 [[-0.3827 -0.9239]
 [-0.9239  0.3827]] 
V^T from R (same up to signs) =
 [[ 0.3827  0.9239]
 [-0.9239  0.3827]]
A^T A =
 [[ 5. 10.]
 [10. 25.]] 
R^T R =
 [[ 5. 10.]
 [10. 25.]]
W^T W =
 [[ 1. -0.]
 [-0.  1.]]
W Sigma V^T =
 [[2.2361 4.4721]
 [0.     2.2361]] 
R =
 [[2.2361 4.4721]
 [0.     2.2361]]


### <span style="color:#1f4e79">10. The SVD Solves $Ax = b$: the Pseudoinverse</span>

* Moore–Penrose pseudoinverse from the SVD:

$$
A^+ = V\,\Sigma^+ U^T, \qquad \Sigma^+ = \text{invert the nonzero } \sigma_i \text{ (zeros stay zero)}, \qquad x^+ = A^+ b
$$

* $x^+$ adapts automatically to the problem:

| Case | Shape | $x^+ = A^+b$ is |
|---|---|---|
| 1 | square, invertible | the **unique exact** solution, $A^{-1}b$ |
| 2 | overdetermined (more equations) | the **least-squares** solution: minimizes $\Vert Ax - b\Vert $ |
| 3 | underdetermined (more unknowns) | the **minimum-norm** solution: smallest $\Vert x\Vert $ among all solutions |

#### <span style="color:#455a64">Example 1: square system (our $A$)</span>

$$
A = \begin{bmatrix} 2 & 3 \\ 1 & 4 \end{bmatrix}, \quad b = \begin{bmatrix} 16 \\ 13 \end{bmatrix}
$$

* $A$ is **square** and has **full rank**: $\det A = 2\cdot4 - 3\cdot1 = 5 \neq 0$, both $\sigma_i > 0$.
* Columns $(2,1)$ and $(3,4)$ are independent ⟹ they reach every $b$, and in only one way.
* So $Ax = b$ has a **unique** solution.
* Square alone is not enough: $\begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}$ is square but has rank 1 ($\sigma_2 = 0$).

**The idea:** undo the three SVD steps in **reverse** order

$$
Ax = b \;\Longrightarrow\; U\Sigma V^T x = b \;\Longrightarrow\; x = V\,\Sigma^{-1}\,U^T b
$$

* Undo a rotation: multiply by its transpose ($U^T$, $V$) — free.
* Undo the stretch: divide by each $\sigma_i$.

#### <span style="color:#455a64">Inverse of a diagonal matrix</span>

$$
D = \begin{bmatrix} d_1 & & \\ & \ddots & \\ & & d_n \end{bmatrix}
\quad\Longrightarrow\quad
D^{-1} = \begin{bmatrix} 1/d_1 & & \\ & \ddots & \\ & & 1/d_n \end{bmatrix}
\qquad \text{if every } d_i \neq 0
$$

* $D$ multiplies coordinate $i$ by $d_i$; dividing by $d_i$ undoes it.
* If **one** $d_i = 0$: no inverse ($\det D = 0$, that direction is squashed to zero).
* "$1/d_i$ for the nonzero $d_i$, zeros stay zero" is the **pseudoinverse** $D^+$, not an inverse.
* Same rule for $\Sigma$ (singular values) and for $\Lambda$ (eigenvalues).

$$
\Sigma^{-1} = \begin{bmatrix} 1/5.398 & 0 \\ 0 & 1/0.926 \end{bmatrix} \approx \begin{bmatrix} 0.185 & 0 \\ 0 & 1.080 \end{bmatrix}
$$

#### <span style="color:#455a64">Solving step by step</span>

1. **Rotate $b$ back:** $c = U^T b$ (length unchanged, $\|c\| = \|b\| = 20.62$)
$$
c = \begin{bmatrix} 0.655(16) + 0.755(13) \\ -0.755(16) + 0.655(13) \end{bmatrix} \approx \begin{bmatrix} 20.304 \\ -3.570 \end{bmatrix}
$$

2. **Undo the stretch:** $y = \Sigma^{-1} c$ (divide by each $\sigma_i$)
$$
y = \begin{bmatrix} 20.304 / 5.398 \\ -3.570 / 0.926 \end{bmatrix} \approx \begin{bmatrix} 3.761 \\ -3.854 \end{bmatrix}
$$

3. **Rotate into place:** $x = V y$ (length unchanged)
$$
x = \begin{bmatrix} 0.383 & -0.924 \\ 0.924 & 0.383 \end{bmatrix} \begin{bmatrix} 3.761 \\ -3.854 \end{bmatrix} \approx \begin{bmatrix} 5 \\ 2 \end{bmatrix}
$$

* The **forward** walk was $x \to x_1 = (3.761, -3.854) \to x_2 = (20.304, -3.570) \to b$.
* Solving visits the **same points in reverse**: $c = x_2$, $y = x_1$.
* No elimination, no inverse matrix: two transposes and one division per $\sigma_i$.
* Same in one line:
$$
x = \sum_i \frac{u_i \cdot b}{\sigma_i}\, v_i
$$
* Small $\sigma_i$ ⟹ dividing **amplifies** errors in $b$ along $u_i$.

In [ ]:
A = np.array([[2.0, 3.0],
              [1.0, 4.0]])
b = np.array([16.0, 13.0])

U, s, Vt = np.linalg.svd(A)
if np.linalg.det(Vt) < 0:          # same sign choice as in Section 8 (U, V rotations)
    U[:, 0] *= -1
    Vt[0, :] *= -1

c = U.T @ b                         # step 1: rotate b back
y = c / s                           # step 2: undo the stretch
x = Vt.T @ y                        # step 3: rotate into place

print("c = U^T b       =", c)
print("y = Sigma^-1 c  =", y)
print("x = V y         =", x)
print("np.linalg.solve =", np.linalg.solve(A, b))
print("Sigma^-1 =\n", np.diag(1/s))

c = U^T b       = [20.3041 -3.5696]
y = Sigma^-1 c  = [ 3.7612 -3.854 ]
x = V y         = [5. 2.]
np.linalg.solve = [5. 2.]
Sigma^-1 =
 [[0.1852 0.    ]
 [0.     1.0797]]


#### <span style="color:#455a64">Example 2: tall system (more equations than unknowns)</span>

$$
\begin{aligned}
x_1 &= 1\\
x_2 &= 2\\
x_1 + x_2 &= 6
\end{aligned}
\qquad\Longleftrightarrow\qquad
\underbrace{\begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix}}_{A\ (3\times 2)}
\begin{bmatrix} x_1 \\ x_2 \end{bmatrix}
=
\underbrace{\begin{bmatrix} 1 \\ 2 \\ 6 \end{bmatrix}}_{b}
$$

* **Full rank:** columns $(1,0,1)$ and $(0,1,1)$ are independent ⟹ $\mathrm{rank}\,A = 2$.
* **No exact solution:** the first two equations force $x_1 + x_2 = 3$, the third asks for $6$.
* Column picture: the two columns span only a **plane** in $\mathbb{R}^3$; $b$ is not in that plane.
* Look instead for the $x$ that brings $Ax$ **closest** to $b$: the **least-squares** solution, $\min \|Ax - b\|$.

#### <span style="color:#455a64">SVD of $A$ (reduced: $\hat U$ is $3 \times 2$)</span>

$$
A^TA = \begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix}\begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}
$$

* Eigenvalues $3$ and $1$ ⟹ $\sigma_1 = \sqrt3$, $\sigma_2 = 1$.
* Eigenvectors: $v_1 = \tfrac{1}{\sqrt2}(1, 1)$, $\;v_2 = \tfrac{1}{\sqrt2}(1, -1)$.
* $u_i = Av_i/\sigma_i$: $\;u_1 = \tfrac{1}{\sqrt6}(1, 1, 2)$, $\;u_2 = \tfrac{1}{\sqrt2}(1, -1, 0)$.

$$
\hat U = \begin{bmatrix} 1/\sqrt6 & 1/\sqrt2 \\ 1/\sqrt6 & -1/\sqrt2 \\ 2/\sqrt6 & 0 \end{bmatrix}, \quad
\Sigma = \begin{bmatrix} \sqrt3 & 0 \\ 0 & 1 \end{bmatrix}, \quad
V = \frac{1}{\sqrt2}\begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}
$$

* $u_1, u_2$: an orthonormal basis of the **plane** spanned by the columns of $A$.

#### <span style="color:#455a64">Solving: $x = V\,\Sigma^{-1}\,\hat U^T b$</span>

1. **Measure $b$ along $u_1, u_2$:** $c = \hat U^T b$
$$
c_1 = u_1\cdot b = \frac{1 + 2 + 12}{\sqrt6} = \frac{15}{\sqrt6}, \qquad
c_2 = u_2\cdot b = \frac{1 - 2}{\sqrt2} = -\frac{1}{\sqrt2}
$$

2. **Undo the stretch:** $y = \Sigma^{-1} c$
$$
y_1 = \frac{15/\sqrt6}{\sqrt3} = \frac{5}{\sqrt2}, \qquad y_2 = \frac{-1/\sqrt2}{1} = -\frac{1}{\sqrt2}
$$

3. **Rebuild along $v_1, v_2$:** $x = V y = y_1 v_1 + y_2 v_2$
$$
x = \frac{5}{2}\begin{bmatrix} 1 \\ 1 \end{bmatrix} - \frac{1}{2}\begin{bmatrix} 1 \\ -1 \end{bmatrix} = \begin{bmatrix} 2 \\ 3 \end{bmatrix}
$$

#### <span style="color:#455a64">Check: how close did we get?</span>

$$
Ax = \begin{bmatrix} 2 \\ 3 \\ 5 \end{bmatrix}, \qquad
r = b - Ax = \begin{bmatrix} -1 \\ -1 \\ 1 \end{bmatrix}, \qquad \|r\| = \sqrt3
$$

* $Ax = (2,3,5)$ is the point of the plane **closest** to $b = (1,2,6)$.
* The residual is **perpendicular** to the plane: $r\cdot(1,0,1) = 0$, $\;r\cdot(0,1,1) = 0$.

#### <span style="color:#455a64">What is new compared with the square case</span>

* $b$ has a third direction $u_3 = \tfrac{1}{\sqrt3}(1, 1, -1)$, perpendicular to the plane: **no $x$ can produce it**.
* Part of $b$ along $u_3$: $\;u_3\cdot b = \dfrac{1 + 2 - 6}{\sqrt3} = -\sqrt3$ ⟹ $r = -\sqrt3\, u_3 = (-1, -1, 1)$: exactly the residual.
* The SVD splits $b$:
  * along $u_1, u_2$: **can** be reached ⟹ solved exactly
  * along $u_3$: **cannot** be reached ⟹ left over as the error
* Same formula and steps; only $\hat U$ is now $3\times2$.

In [ ]:
A_tall = np.array([[1.0, 0.0],
                   [0.0, 1.0],
                   [1.0, 1.0]])
b_tall = np.array([1.0, 2.0, 6.0])

U, s, Vt = np.linalg.svd(A_tall)          # full U is 3 x 3; columns 1-2 are U_hat
c = U[:, :2].T @ b_tall                    # step 1
y = c / s                                  # step 2
x = Vt.T @ y                               # step 3
r = b_tall - A_tall @ x

print("sigma =", s)
print("x (SVD)   =", x, "   lstsq gives", np.linalg.lstsq(A_tall, b_tall, rcond=None)[0])
print("A x       =", A_tall @ x)
print("r = b - Ax =", r, "   |r| =", np.linalg.norm(r))
print("A^T r     =", A_tall.T @ r, "  (r is perpendicular to the columns)")
print("u3 . b    =", U[:, 2] @ b_tall, "  (NumPy may flip the sign of u3)")

sigma = [1.7321 1.    ]
x (SVD)   = [2. 3.]    lstsq gives [2. 3.]
A x       = [2. 3. 5.]
r = b - Ax = [-1. -1.  1.]    |r| = 1.7320508075688774
A^T r     = [0. 0.]   (r is perpendicular to the columns)
u3 . b    = 1.732050807568877   (NumPy may flip the sign of u3)


#### <span style="color:#455a64">Example 3: wide system (fewer equations than unknowns)</span>

$$
\begin{aligned}
x_1 \phantom{{}+x_2} + x_3 &= 2\\
x_2 + x_3 &= 4
\end{aligned}
\qquad\Longleftrightarrow\qquad
\underbrace{\begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix}}_{A\ (2\times 3)}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}
=
\underbrace{\begin{bmatrix} 2 \\ 4 \end{bmatrix}}_{b}
$$

* $A$ is the **transpose** of Example 2: the same $\sqrt2, \sqrt3, \sqrt6$ come back.
* **Full rank:** $\mathrm{rank}\,A = 2$ ⟹ the columns fill all of $\mathbb{R}^2$ ⟹ a solution always exists.
* **Infinitely many solutions:** 3 unknowns, 2 equations ⟹ one direction is free.

#### <span style="color:#455a64">SVD of $A$, from $A^TA$ ($3\times3$)</span>

$$
A^TA = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix}\begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix} = \begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \\ 1 & 1 & 2 \end{bmatrix}
$$

* Characteristic equation: $\lambda^3 - 4\lambda^2 + 3\lambda = \lambda(\lambda-1)(\lambda-3) = 0$ ⟹ $\lambda = 3, 1, 0$.
* $\sigma_1 = \sqrt3$, $\;\sigma_2 = 1$, $\;\sigma_3 = 0$.
* Eigenvectors of $A^TA$ (right singular vectors, in 3-D):
  * $\lambda = 3$: $\;v_1 = \tfrac{1}{\sqrt6}(1, 1, 2)$
  * $\lambda = 1$: $\;v_2 = \tfrac{1}{\sqrt2}(1, -1, 0)$
  * $\lambda = 0$: $\;v_3 = \tfrac{1}{\sqrt3}(1, 1, -1)$ — the free direction $n$: $A$ sends it to zero
* $u_i = Av_i/\sigma_i$: $\;u_1 = \tfrac{1}{\sqrt2}(1, 1)$, $\;u_2 = \tfrac{1}{\sqrt2}(1, -1)$.
* $v_3$: $\sigma_3 = 0$, we **cannot divide** ⟹ no $u_3$ (and $A$ has only 2 rows anyway).

$$
U = \frac{1}{\sqrt2}\begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}, \quad
\Sigma = \begin{bmatrix} \sqrt3 & 0 \\ 0 & 1 \end{bmatrix}, \quad
\hat V = \begin{bmatrix} 1/\sqrt6 & 1/\sqrt2 \\ 1/\sqrt6 & -1/\sqrt2 \\ 2/\sqrt6 & 0 \end{bmatrix}
$$

* Shortcut: $AA^T$ ($2\times2$) has the same nonzero eigenvalues ($3, 1$) and gives $U$ directly; then $v_i = A^Tu_i/\sigma_i$. Same SVD, but $v_3$ does not appear.

#### <span style="color:#455a64">Solving: $x = \hat V\,\Sigma^{-1}\,U^T b$</span>

1. **Measure $b$ along $u_1, u_2$:** $c = U^T b$
$$
c_1 = u_1\cdot b = \frac{2 + 4}{\sqrt2} = 3\sqrt2, \qquad
c_2 = u_2\cdot b = \frac{2 - 4}{\sqrt2} = -\sqrt2
$$

2. **Undo the stretch:** $y = \Sigma^{-1} c$
$$
y_1 = \frac{3\sqrt2}{\sqrt3} = \sqrt6, \qquad y_2 = \frac{-\sqrt2}{1} = -\sqrt2
$$
* $\sigma_3 = 0$: nothing to divide ⟹ **no $y_3$**, nothing along $v_3$.

3. **Rebuild along $v_1, v_2$ only:** $x = y_1 v_1 + y_2 v_2$
$$
x = \sqrt6\cdot\frac{1}{\sqrt6}\begin{bmatrix} 1 \\ 1 \\ 2 \end{bmatrix} - \sqrt2\cdot\frac{1}{\sqrt2}\begin{bmatrix} 1 \\ -1 \\ 0 \end{bmatrix}
= \begin{bmatrix} 1 \\ 1 \\ 2 \end{bmatrix} - \begin{bmatrix} 1 \\ -1 \\ 0 \end{bmatrix} = \begin{bmatrix} 0 \\ 2 \\ 2 \end{bmatrix}
$$

4. **Residual:**
$$
Ax = \begin{bmatrix} 0 + 0 + 2 \\ 0 + 2 + 2 \end{bmatrix} = \begin{bmatrix} 2 \\ 4 \end{bmatrix} = b
\qquad\Longrightarrow\qquad r = b - Ax = 0
$$

* $x$ solves the system **exactly**, and $\|x\| = \sqrt8$: the **shortest** solution.

#### <span style="color:#455a64">Why the shortest solution?</span>

* $x$ is built from $v_1, v_2$ only ⟹ $x \perp v_3$: $\;(0,2,2)\cdot(1,1,-1) = 0$.
* Every other solution adds $t\,n$ at a right angle ⟹ Pythagoras:
$$
\|x + t\,n\|^2 = \|x\|^2 + t^2\|n\|^2 = 8 + 3t^2 \qquad (\text{smallest at } t = 0)
$$

* Picture: the solutions form a **line** in 3-D; the SVD answer is the point on the line **closest to the origin**.
* Why the residual is zero here but not in Example 2:
  * **tall:** $b$ lives in 3-D, $A$ reaches only a plane ⟹ part of $b$ is left over ⟹ $r \neq 0$
  * **wide:** $b$ lives in 2-D, $A$ (rank 2) reaches all of it ⟹ $r = 0$; instead $x$ has a **free direction**

#### <span style="color:#455a64">The three cases side by side: one formula $x = V\Sigma^+U^Tb$</span>

| Case | $A$ | Solutions | What the SVD gives |
|---|---|---|---|
| square | $2\times2$, full rank | exactly one | that solution, $x = (5, 2)$ |
| tall | $3\times2$, full rank | none (exact) | **least squares**: closest $Ax$ to $b$, $x = (2, 3)$, $\Vert r\Vert  = \sqrt3$ |
| wide | $2\times3$, full rank | infinitely many | **minimum norm**: shortest $x$, $x = (0, 2, 2)$, $r = 0$ |

| Case | leftover direction | lives in | what the SVD does |
|---|---|---|---|
| tall | $u_3 = \tfrac{1}{\sqrt3}(1,1,-1)$ | $b$ (output) | drops $b$'s part along $u_3$ ⟹ residual |
| wide | $v_3 = \tfrac{1}{\sqrt3}(1,1,-1)$ | $x$ (input) | adds nothing along $v_3$ ⟹ shortest $x$ |

* The SVD never divides by a zero (or missing) $\sigma$

#### <span style="color:#455a64">One function for all cases</span>

* The same formula as a function, `pinv_svd`, tried on three more examples.

In [ ]:
def pinv_svd(A, tol=1e-12):
    "Pseudoinverse A^+ = V Sigma^+ U^T from the reduced SVD."
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    s_plus = np.array([1/si if si > tol*s[0] else 0.0 for si in s])
    return Vt.T @ np.diag(s_plus) @ U.T

# Case 1: square, invertible (our A)
b = np.array([16.0, 13.0])
print("Case 1:  x+ =", pinv_svd(A) @ b, "   solve gives", np.linalg.solve(A, b))

# Case 2: overdetermined -- line y = c0 + c1 t through 3 points that are not collinear
t, y = np.array([0.0, 1.0, 2.0]), np.array([1.0, 2.0, 2.0])
A2 = np.column_stack([np.ones(3), t])
c = pinv_svd(A2) @ y
print("Case 2:  x+ =", c, "   lstsq gives", np.linalg.lstsq(A2, y, rcond=None)[0])

# Case 3: underdetermined -- one equation x1 + 2 x2 = 5
A3, b3 = np.array([[1.0, 2.0]]), np.array([5.0])
x_plus = pinv_svd(A3) @ b3
print("Case 3:  x+ =", x_plus, "   |x+| =", np.linalg.norm(x_plus), "   other solution (5,0) has |x| = 5")

Case 1:  x+ = [5. 2.]    solve gives [5. 2.]
Case 2:  x+ = [1.1667 0.5   ]    lstsq gives [1.1667 0.5   ]
Case 3:  x+ = [1. 2.]    |x+| = 2.2360679774997894    other solution (5,0) has |x| = 5


#### <span style="color:#455a64">Visualization: the minimum-norm solution (Case 3)</span>

In [ ]:
def min_norm_picture(b=5.0, s=1.5):
    a = np.array([1.0, 2.0])                        # the equation a . x = b
    x_plus = b * a / (a @ a)                         # = A^+ b
    d = np.array([-a[1], a[0]]) / np.linalg.norm(a)  # direction along the line
    x_any = x_plus + s * d
    C = unit_circle()

    fig, ax = plt.subplots(figsize=(6.8, 6.5))
    ax.plot(*np.array([x_plus - 10*d, x_plus + 10*d]).T, color=BLUE, lw=2, label=r"all solutions of $x_1 + 2x_2 = b$")
    r = np.linalg.norm(x_plus)
    ax.plot(r*C[0], r*C[1], color=GRAY, ls="--", lw=1)
    arrow(ax, x_any, color=ORANGE, lw=2.5, label="$x$", offset=(-0.5, 0.2))
    arrow(ax, x_plus, color=RED, lw=3, label="$x^+$", offset=(0.3, -0.8))
    ax.legend(loc="lower left", fontsize=9)
    setup(ax, (-6, 6), (-6, 6),
          f"x+ = A+ b = ({x_plus[0]:.2f}, {x_plus[1]:.2f}),  |x+| = {r:.2f}\n"
          f"orange: another solution, |x| = {np.linalg.norm(x_any):.2f}")
    plt.show()

interact(min_norm_picture,
         b=FloatSlider(value=5, min=-6, max=6, step=0.5, description="b"),
         s=FloatSlider(value=1.5, min=-4, max=4, step=0.25, description="s"));

interactive(children=(FloatSlider(value=5.0, description='b', max=6.0, min=-6.0, step=0.5), FloatSlider(value=…

* One equation, two unknowns: $\;x_1 + 2x_2 = b$ ⟹ a whole **line** of solutions.
* Slider $b$ moves the line; slider $s$ moves a point along the line (any solution).
* $x^+$ (red) is the solution **closest to the origin**: it is perpendicular to the line.
* The dashed circle of radius $\|x^+\|$ just touches the line.

### <span style="color:#1f4e79">Summary</span>

#### <span style="color:#455a64">What we saw</span>

* $Ax$ = combination of the columns of $A$; the entries of $x$ are the weights.
* Orthogonal columns ⟹ coordinates by projection, $x_i = \dfrac{b\cdot v_i}{\|v_i\|^2}$.
* Orthonormal columns ⟹ $x_i = b\cdot q_i$, i.e. $x = A^Tb$; $A^TA = I$ always, $A^{-1} = A^T$ only if square.
* Orthogonal matrices preserve lengths and angles: rotations and reflections.
* A general matrix stretches **and** rotates; the stretch depends on the direction.
* SVD: $A = U\Sigma V^T$ = rotate → stretch → rotate; $Av_i = \sigma_iu_i$.
* QR: $A = QR$ = perpendicular frame $Q$ + triangular coordinates $R$ (Gram–Schmidt); $A$ and $R$ have the same $\sigma_i$.
* Pseudoinverse: $x^+ = V\Sigma^+U^Tb$ gives the exact, least-squares or minimum-norm solution.

#### <span style="color:#455a64">Practical importance: SVD vs QR</span>

| | **SVD** $\;A = U\Sigma V^T$ | **QR** $\;A = QR$ |
|---|---|---|
| geometry | rotate → stretch → rotate | perpendicular frame + triangular coordinates |
| exists for | every matrix | every matrix |
| main uses | rank, norms, condition number; pseudoinverse; low-rank approximation / compression; rank-deficient least squares | fast, stable least squares; QR algorithm for eigenvalues; orthonormal bases (Krylov methods) |
| cost | higher | lower |
| algorithms (How it is computed)| Householder bidiagonalization + QR iteration (Golub–Kahan–Reinsch); divide-and-conquer | Householder reflections; Givens rotations; modified Gram–Schmidt |